# LightGBM Training 



What it does:
- uses the shared raw loaders, feature builder, WMAE metric, and expanding-window validation;
- keeps LightGBM categorical columns native (`Store`, `Dept`, `Type`, `HolidayName`);
- trains with holiday sample weights so the objective matches WMAE better;
- keeps the required MLflow stages: cleaning, feature selection, cross-validation, HPO, and final model;
- saves the final model as an sklearn `Pipeline` that accepts raw Walmart rows.

Do not run all cells blindly. Fill credentials in `.env`, review paths, then run stage by stage when ready.

In [2]:
import sys
print(sys.executable)

c:\Users\User\Documents\freeuni\ml\ML-final\.venv\Scripts\python.exe


In [3]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

warnings.filterwarnings("ignore")
DATA_DIR = ROOT / "data"
MODEL_NAME = "LightGBM"
EXPERIMENT_NAME = "LightGBM_Training"
RANDOM_STATE = 42

In [4]:
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

import lightgbm as lgb

try:
    import optuna
except ImportError:
    optuna = None

mlflow.set_experiment(EXPERIMENT_NAME)

# Run this cell once before the stage cells when you want MLflow to group
# the required Cleaning -> Feature Selection -> CV -> HPO -> Final runs.
parent_run = mlflow.start_run(run_name="LightGBM_Workflow")

## Load Raw Data

The model always starts from raw competition files. Feature engineering happens through `merge_all` and `build_features`, which keeps this notebook aligned with the shared project code.

In [5]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)
train_raw = train_raw.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

target = train_raw["Weekly_Sales"].astype(float)
holiday_weights = np.where(train_raw["IsHoliday"].astype(bool), 5.0, 1.0)

In [6]:
CATEGORICAL_COLUMNS = ["Store", "Dept", "Type", "HolidayName"]

def _lightgbm_categoricals(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for col in CATEGORICAL_COLUMNS:
        if col in X.columns:
            X[col] = X[col].astype("category")
    return X

def make_lgb_features(raw_frame: pd.DataFrame, sales_history: pd.DataFrame | None = None) -> pd.DataFrame:
    merged = merge_all(raw_frame, features_df, stores_df)
    history_merged = None if sales_history is None else merge_all(sales_history, features_df, stores_df)
    X = build_features(merged, sales_history_df=history_merged, encode_categoricals=False)
    return _lightgbm_categoricals(X)

def align_columns(X: pd.DataFrame, reference_columns: list[str]) -> pd.DataFrame:
    X = X.reindex(columns=reference_columns, fill_value=0)
    return _lightgbm_categoricals(X)

def make_sample_weight(frame: pd.DataFrame) -> np.ndarray:
    return np.where(frame["IsHoliday"].astype(bool), 5.0, 1.0)

def default_lgbm(**overrides):
    params = {
        "objective": "l1",
        "n_estimators": 800,
        "learning_rate": 0.04,
        "num_leaves": 96,
        "min_child_samples": 40,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
    }
    params.update(overrides)
    return lgb.LGBMRegressor(**params)

## MLflow Stage: Cleaning

The shared feature builder fills markdown NaNs with zero and keeps missing flags. Negative training sales are kept, because they represent returns; predictions are clipped at zero only when creating a Kaggle submission.

In [7]:
with mlflow.start_run(run_name="LightGBM_Cleaning", nested=True):
    mlflow.log_param("negative_sales_training_policy", "keep")
    mlflow.log_param("negative_prediction_policy", "clip_to_zero_for_submission")
    mlflow.log_param("markdown_nan_policy", "fill_zero_plus_missing_flags_in_features_py")
    mlflow.log_metric("negative_sales_rows", int((train_raw["Weekly_Sales"] < 0).sum()))
    mlflow.log_metric("train_rows", int(len(train_raw)))
    mlflow.log_metric("test_rows", int(len(test_raw)))

🏃 View run LightGBM_Cleaning at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1/runs/5e3ed0fecd2544488368aa3348608277
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1


## MLflow Stage: Feature Selection

Fit one reasonable LightGBM model, inspect feature importance, and optionally drop features with zero gain. Keep the dropped feature list as an MLflow artifact so the README can explain the choice.

In [8]:
def select_features(min_gain: float = 0.0) -> list[str]:
    X = make_lgb_features(train_raw)
    model = default_lgbm(n_estimators=300)
    cat_cols = [c for c in CATEGORICAL_COLUMNS if c in X.columns]
    model.fit(X, target, sample_weight=holiday_weights, categorical_feature=cat_cols)
    importance = pd.DataFrame({"feature": X.columns, "gain": model.booster_.feature_importance(importance_type="gain")})
    selected = importance.loc[importance["gain"] > min_gain, "feature"].tolist()
    if not selected:
        selected = X.columns.tolist()
    importance.sort_values("gain", ascending=False).to_csv(ROOT / "models" / "lightgbm_feature_importance.csv", index=False)
    return selected

with mlflow.start_run(run_name="LightGBM_Feature_Selection", nested=True):
    selected_features = select_features(min_gain=0.0)
    mlflow.log_param("feature_selection_rule", "drop_zero_gain_features")
    mlflow.log_metric("selected_feature_count", len(selected_features))
    mlflow.log_artifact(str(ROOT / "models" / "lightgbm_feature_importance.csv"))

🏃 View run LightGBM_Feature_Selection at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1/runs/6c0f60302a964ec694651414fa3316ea
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1


## MLflow Stage: Cross-Validation

Validation splits are made by date, not by row. Each fold trains on the past and validates on the next 39 weeks, matching the real Kaggle horizon.

In [9]:
def cv_lightgbm(params: dict | None = None, selected_columns: list[str] | None = None) -> list[float]:
    params = params or {}
    scores = []
    for fold, (tr_mask, val_mask) in enumerate(expanding_window_folds(train_raw["Date"], n_splits=3, horizon=39), start=1):
        fold_train = train_raw.loc[tr_mask].copy()
        fold_val = train_raw.loc[val_mask].copy()
        X_train = make_lgb_features(fold_train)
        if selected_columns is None:
            selected_columns = X_train.columns.tolist()
        X_train = align_columns(X_train, selected_columns)
        X_val = align_columns(make_lgb_features(fold_val, sales_history=fold_train), selected_columns)
        model = default_lgbm(**params)
        cat_cols = [c for c in CATEGORICAL_COLUMNS if c in X_train.columns]
        model.fit(
            X_train,
            fold_train["Weekly_Sales"].astype(float),
            sample_weight=make_sample_weight(fold_train),
            categorical_feature=cat_cols,
        )
        pred = np.clip(model.predict(X_val), 0, None)
        score = wmae(fold_val["Weekly_Sales"], pred, fold_val["IsHoliday"])
        scores.append(score)
    return scores

with mlflow.start_run(run_name="LightGBM_CrossValidation", nested=True):
    cv_scores = cv_lightgbm(selected_columns=selected_features)
    for i, score in enumerate(cv_scores, start=1):
        mlflow.log_metric(f"fold_{i}_wmae", score)
    mlflow.log_metric("mean_cv_wmae", float(np.mean(cv_scores)))

🏃 View run LightGBM_CrossValidation at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1/runs/6d2f362e2ed5458bbc3bc4bd800de0a7
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1


## MLflow Stage: HPO

Optuna searches the main LightGBM capacity and regularization parameters. Keep `N_TRIALS` modest first, then increase only if the CV loop is stable.

In [10]:
N_TRIALS = 25

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 120),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.65, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.65, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 400, 1400),
    }
    scores = cv_lightgbm(params=params, selected_columns=selected_features)
    return float(np.mean(scores))

with mlflow.start_run(run_name="LightGBM_HPO", nested=True):
    if optuna is None:
        raise ImportError("Install optuna before running HPO: pip install optuna")
    study = optuna.create_study(direction="minimize", study_name="lightgbm_wmae")
    study.optimize(objective, n_trials=N_TRIALS)
    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.log_metric("best_mean_cv_wmae", study.best_value)

[I 2026-07-12 01:40:11,171] A new study created in memory with name: lightgbm_wmae
[I 2026-07-12 01:44:47,530] Trial 0 finished with value: 2094.593027189837 and parameters: {'num_leaves': 181, 'learning_rate': 0.025660491364016734, 'min_child_samples': 61, 'feature_fraction': 0.8043053866161021, 'bagging_fraction': 0.7600534521309398, 'n_estimators': 994}. Best is trial 0 with value: 2094.593027189837.
[I 2026-07-12 01:51:14,677] Trial 1 finished with value: 2106.051538752871 and parameters: {'num_leaves': 190, 'learning_rate': 0.011797350183652789, 'min_child_samples': 85, 'feature_fraction': 0.7387943842012438, 'bagging_fraction': 0.9518671999022792, 'n_estimators': 1117}. Best is trial 0 with value: 2094.593027189837.
[I 2026-07-12 01:55:14,552] Trial 2 finished with value: 2082.7359731638458 and parameters: {'num_leaves': 202, 'learning_rate': 0.03171998621927002, 'min_child_samples': 97, 'feature_fraction': 0.7220345626950133, 'bagging_fraction': 0.9359710677642734, 'n_estimators

🏃 View run LightGBM_HPO at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1/runs/d4a4a566bff5449cbb09d97ee2da41be
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/1


KeyboardInterrupt: 

## MLflow Stage: Final Pipeline

The final object is an sklearn `Pipeline`: raw rows in, feature matrix inside the preprocessor, predictions out. If the team later adds a shared `pipeline.py`, replace the local class with that shared implementation.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, features_df, stores_df, encode_categoricals=False):
        self.features_df = features_df
        self.stores_df = stores_df
        self.encode_categoricals = encode_categoricals

    def fit(self, X, y=None):
        raw = X.copy()
        if y is not None:
            raw = raw.assign(Weekly_Sales=np.asarray(y, dtype=float))
        self.history_ = merge_all(raw, self.features_df, self.stores_df)
        fitted = build_features(self.history_, encode_categoricals=self.encode_categoricals)
        self.feature_columns_ = fitted.columns.tolist()
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        return self._transform_train(X, y)

    def _transform_train(self, X, y=None):
        raw = X.copy()
        if y is not None:
            raw = raw.assign(Weekly_Sales=np.asarray(y, dtype=float))
        merged = merge_all(raw, self.features_df, self.stores_df)
        X_feat = build_features(merged, encode_categoricals=self.encode_categoricals)
        X_feat = X_feat.reindex(columns=self.feature_columns_, fill_value=0)
        return _lightgbm_categoricals(X_feat)

    def transform(self, X):
        merged = merge_all(X.copy(), self.features_df, self.stores_df)
        X_feat = build_features(merged, sales_history_df=self.history_, encode_categoricals=self.encode_categoricals)
        X_feat = X_feat.reindex(columns=self.feature_columns_, fill_value=0)
        return _lightgbm_categoricals(X_feat)

In [ ]:
with mlflow.start_run(run_name="LightGBM_Final", nested=True):
    final_params = globals().get("best_params", {})
    final_model = default_lgbm(**final_params)
    final_pipeline = Pipeline([
        ("prep", WalmartPreprocessor(features_df=features_df, stores_df=stores_df, encode_categoricals=False)),
        ("model", final_model),
    ])
    final_pipeline.fit(train_raw.drop(columns=["Weekly_Sales"]), train_raw["Weekly_Sales"], model__sample_weight=holiday_weights)
    mlflow.log_params(final_params)
    mlflow.sklearn.log_model(final_pipeline, artifact_path="model")

    test_pred = np.clip(final_pipeline.predict(test_raw), 0, None)
    submission = make_submission(test_raw, test_pred, ROOT / "models" / "lightgbm_submission.csv")
    mlflow.log_artifact(str(ROOT / "models" / "lightgbm_submission.csv"))

mlflow.end_run()

## README Takeaways

- LightGBM is expected to be the strongest Ketevan model because Walmart sales contain many categorical interactions: store, department, store type, holiday, and markdown behavior.
- The model is optimized against WMAE by using an L1 objective and holiday sample weights of 5.
- Feature importance should be used to explain whether markdowns, holiday flags, and year-ago sales history actually helped.